# 🎬 Studio Video AI - Gerador Direto de Vídeo com Sincronia Labial (SadTalker 512px + GFPGAN)
### Execução 100% gratuita na GPU NVIDIA T4 do Google Colab

**Instruções:**
1. Verifique se o ambiente está em **GPU T4** (Ambiente de execução -> Alterar tipo de ambiente -> GPU T4).
2. Clique no **Play ▶** abaixo.
3. O processo leva cerca de **1 minuto e meio**. Ao finalizar, ele vai parar sozinho e **baixar o vídeo MP4 realista direto para o seu computador**!

In [ ]:
# ==============================================================================
# 🚀 RENDERIZADOR NEURAL DIRETO (SADTALKER 512PX + GFPGAN)
# ==============================================================================
import os, sys, glob

print('📦 1. Verificando dependências neurais...')
!pip install -q basicsr-fixed
!pip install -q kornia facexlib yacs safetensors gradio edge-tts
!pip install -q --no-build-isolation basicsr
!pip install -q gfpgan

%cd /content
if not os.path.exists('/content/SadTalker'):
    !git clone -b v1.1 https://github.com/camenduru/SadTalker

%cd /content/SadTalker

# 2. Correção definitiva do NumPy em Python
preprocess_file = '/content/SadTalker/src/face3d/util/preprocess.py'
if os.path.exists(preprocess_file):
    with open(preprocess_file, 'r') as f:
        content = f.read()
    content = content.replace('np.VisibleDeprecationWarning', 'UserWarning')
    with open(preprocess_file, 'w') as f:
        f.write(content)

# 3. Baixa/Verifica pesos neurais essenciais
!apt -y install -qq aria2
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/SadTalker_V0.0.2_256.safetensors -d /content/SadTalker/checkpoints -o SadTalker_V0.0.2_256.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/SadTalker_V0.0.2_512.safetensors -d /content/SadTalker/checkpoints -o SadTalker_V0.0.2_512.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/mapping_00109-model.pth.tar -d /content/SadTalker/checkpoints -o mapping_00109-model.pth.tar
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/mapping_00229-model.pth.tar -d /content/SadTalker/checkpoints -o mapping_00229-model.pth.tar
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/gfpgan/weights/GFPGANv1.4.pth -d /content/SadTalker/gfpgan/weights -o GFPGANv1.4.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/gfpgan/weights/alignment_WFLW_4HG.pth -d /content/SadTalker/gfpgan/weights -o alignment_WFLW_4HG.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/gfpgan/weights/detection_Resnet50_Final.pth -d /content/SadTalker/gfpgan/weights -o detection_Resnet50_Final.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/gfpgan/weights/parsing_parsenet.pth -d /content/SadTalker/gfpgan/weights -o parsing_parsenet.pth

# 4. Baixa a foto do personagem maçônico e o áudio oficial
!curl -s -L -o /content/personagem_maconaria.jpg https://raw.githubusercontent.com/grego921/studio-video-ai/main/personagem_maconaria.jpg
!curl -s -L -o /content/fala_maconica.mp3 https://raw.githubusercontent.com/grego921/studio-video-ai/main/fala_maconica.mp3

# 5. RENDERIZAÇÃO NEURAL DIRETA (1 a 2 minutos)
print('\n' + '='*70)
print('🎬 INICIANDO RENDERIZAÇÃO NEURAL 3D (SadTalker 512px + GFPGAN)...')
print('='*70 + '\n')

!python inference.py \
  --driven_audio /content/fala_maconica.mp3 \
  --source_image /content/personagem_maconaria.jpg \
  --result_dir /content/resultado \
  --still \
  --preprocess crop \
  --enhancer gfpgan \
  --size 512 \
  --batch_size 2

# 6. Baixa automaticamente o vídeo MP4 finalizado para o seu computador
from google.colab import files
videos = glob.glob('/content/resultado/*/*.mp4')
if videos:
    latest_video = max(videos, key=os.path.getctime)
    print('\n' + '='*70)
    print(f'🎉 VÍDEO CONCLUÍDO COM SUCESSO: {latest_video}')
    print('📥 O download do vídeo MP4 começará automaticamente no seu navegador...')
    print('='*70 + '\n')
    files.download(latest_video)
else:
    print('❌ Erro: O vídeo não foi encontrado na pasta de resultados.')
